In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns



def plot_target_distribution(y_array, title="Target Variable Distribution"):
    """
    Plots a histogram and KDE of a numpy array to visualize its distribution.
    """
    # Set the style for a cleaner look
    sns.set_theme(style="whitegrid")

    # Create the figure
    plt.figure(figsize=(10, 6))

    # Plot the histogram with KDE
    sns.histplot(y_array,
                 kde=True,  # Adds the smooth line
                 bins=30,  # Adjust number of bins based on your data size
                 color="steelblue",
                 edgecolor="black",
                 alpha=0.7)

    # Add labels and title
    plt.title(title, fontsize=16, fontweight='bold')
    plt.xlabel("Target Value", fontsize=14)
    plt.ylabel("Frequency", fontsize=14)

    # Add vertical lines for Mean and Median to help spot skewness
    plt.axvline(np.mean(y_array), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(y_array):.2f}')
    plt.axvline(np.median(y_array), color='green', linestyle='-', linewidth=2,
                label=f'Median: {np.median(y_array):.2f}')

    plt.legend()
    plt.tight_layout()

    # Show the plot
    plt.savefig(f'{title}.png')


def plot_regression_diagnostics(y_true, y_pred, title="Model Diagnostics"):
    """
    Generates three standard regression diagnostic plots:
    1. True vs. Predicted Scatter Plot
    2. Residual Plot (Errors vs. Predicted)
    3. Error Distribution (Histogram)
    """
    # Calculate the errors (residuals)
    residuals = y_true - y_pred

    # Set the style
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # --- Plot 1: True vs Predicted Scatter ---
    axes[0].scatter(y_true, y_pred, alpha=0.6, color='steelblue', edgecolor='w', s=60)

    # Plot the ideal "Perfect Prediction" line (y = x)
    min_val = min(np.min(y_true), np.min(y_pred))
    max_val = max(np.max(y_true), np.max(y_pred))
    axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal Fit')

    axes[0].set_title('True vs. Predicted Values')
    axes[0].set_xlabel('True Values')
    axes[0].set_ylabel('Predicted Values')
    axes[0].legend()

    # --- Plot 2: Residuals vs Predicted ---
    # Shows if the error variance changes (heteroscedasticity) or if there's bias
    axes[1].scatter(y_pred, residuals, alpha=0.6, color='darkorange', edgecolor='w', s=60)
    axes[1].axhline(y=0, color='r', linestyle='--', lw=2)

    axes[1].set_title('Residuals vs. Predicted Values')
    axes[1].set_xlabel('Predicted Values')
    axes[1].set_ylabel('Residuals (True - Pred)')

    plt.tight_layout()
    plt.savefig(f'{title}.pdf')

In [ ]:
import os
import pickle

import joblib
import sklearn
import numpy as np

datasets_names = {
        'cnohf_ecfp': ['../results/cnohf_data/cnohf_ecfp/explanations/', '../results/cnohf_data/cnohf_ecfp/', 'detonation_velocity'],
        'cof_ecfp_descriptor': ['../results/cof_data/cof_ecfp_descriptor/explanations/', '../results/cof_data/cof_ecfp_descriptor/', 'capacity_max'],
        'photoswitch_ecfp': ['../results/photoswitch_data/photoswitch_ecfp/explanations/', '../results/photoswitch_data/photoswitch_ecfp/', 'e_isomer_pi_pi'],
        'polymers_ecfp': ['../results/polymers_data/polymers_ecfp/explanations/', '../results/polymers_data/polymers_ecfp/', 'Tg'],
        'redox_ecfp': ['../results/redox_data/redox_ecfp/explanations/', '../results/redox_data/redox_ecfp/', 'dGox'],
        'herg_ecfp_linear': ['../results/synthetic_data/herg_ecfp_linear/explanations/', '../results/synthetic_data/herg_ecfp_linear/', 'target'],
        'herg_ecfp_nonlinear': ['../results/synthetic_data/herg_ecfp_nonlinear/explanations/', '../results/synthetic_data/herg_ecfp_nonlinear/', 'target'],
        'herg_ecfp_piecewise': ['../results/synthetic_data/herg_ecfp_piecewise/explanations/', '../results/synthetic_data/herg_ecfp_piecewise/', 'target'],
    }

results_dict = {
    'lime': 'lime_results.pickle',
    'shap': 'shap_results.pickle',
    'shapiq1': 'shapiq1_results.pickle',
    'shapiq2': 'shapiq2_results.pickle',
    'meg': 'meg2_results.pickle',
    'mmace': 'mmace_results.pickle',
}

r2_datasets = {}

for dataset_name in datasets_names.keys():
    results_dir, model_dir, target = datasets_names[dataset_name]
    print(f"Processing dataset: {dataset_name}")
    key = 'lime'
    r2s = []
    file_name = results_dict[key]
    with open(os.path.join(results_dir, file_name), 'rb') as f:
        results = pickle.load(f)

    for i in range(len(results['test_data'])):
        print(key, i)
        model = joblib.load(os.path.join(model_dir, f'model_{i}.joblib'))
        test_examples = results['test_data'][i].drop(columns=[target])
        y_test = results['test_data'][i][[target]].values.flatten()
        y_train = results['training_data'][i][[target]].values.flatten()

        y_pred = model.predict(test_examples.values)

        r2 = sklearn.metrics.r2_score(y_test, y_pred)
        r2s.append(r2)

    r2_datasets[dataset_name] = r2s

In [ ]:
for dataset_name in datasets_names.keys():
    r2s = r2_datasets[dataset_name]
    mean_r2 = np.mean(r2s)
    std_r2 = np.std(r2s)
    print(f"Dataset: {dataset_name}, Mean, std R^2: {round(mean_r2, 2)}({round(std_r2, 2)})")
    r2_round = [round(r, 2) for r in r2s]
    print(r2_round)

In [ ]:
#polymers outlier fold
from predictive_model.evaluation import EvalMetrics

key = 'lime'
dataset_name = 'polymers_ecfp'
results_dir, model_dir, target = datasets_names['polymers_ecfp']
file_name = results_dict[key]
with open(os.path.join(results_dir, file_name), 'rb') as f:
    results = pickle.load(f)
i = 4
model = joblib.load(os.path.join(model_dir, f'model_{i}.joblib'))
test_examples = results['test_data'][i].drop(columns=[target])
y_test = results['test_data'][i][[target]].values.flatten()
y_train = results['training_data'][i][[target]].values.flatten()

y_pred = model.predict(test_examples.values)

plot_regression_diagnostics(y_test, y_pred, title=f"outlier_polymers")

smape = EvalMetrics().evaluate('smape', y_test, y_pred)
rmse = EvalMetrics().evaluate('rmse', y_test, y_pred)
pa = EvalMetrics().evaluate('pairwise_accuracy_score', y_test, y_pred)

print(f'SMAPE: {smape}')
print(f'RMSE: {rmse}')
print(f'PA: {pa}')